# RAG Evaluation — Retrieval (nDCG, MRR) + Generation (LLM judge)

Two completely separate eval tracks:

| Track | Metrics | Needs | Cost |
|---|---|---|---|
| **Retrieval** | MRR, nDCG@k, Hit@k, Recall@k | Gold chunk per query (auto-generated) | cheap — pure math |
| **Generation** | Faithfulness, Answer relevancy, Context precision | LLM judge | API calls |

**Key insight:** these measure different failure modes. A system can score 0.9 faithfulness
while retrieval recall quietly rots — that's why the tracks are kept separate.

### How gold labels work
MRR and nDCG need to know the *correct* chunk per query. We generate that automatically:
for each chunk, ask the LLM to write one question only that chunk answers → that chunk
becomes the gold doc for that query by construction. No manual labelling needed.

In [1]:
import os, json, math, logging, time
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)
load_dotenv(dotenv_path=os.path.join("..", ".env"))
oai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "aethon_kb"
EMBED_MODEL     = "text-embedding-3-small"   # MUST match ingest
LLM_MODEL       = "gpt-4o-mini"
TESTSET_FILE    = "./testset.json"           # gold labels cached here
TOP_K           = 10
EVAL_K          = 5                          # k for nDCG@k, Recall@k, Hit@k

chroma     = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma.get_collection(name=COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' — {collection.count()} chunks")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Collection 'aethon_kb' — 31 chunks


## 1 — Load chunks from cache

In [2]:
with open("./chunks_cache.json", encoding="utf-8") as f:
    cache = json.load(f)

chunks = [{k: v for k, v in c.items() if k != "embedding"} for c in cache]
print(f"Loaded {len(chunks)} chunks")
print(f"Sample: {chunks[0]['chunk_id']} — {chunks[0]['topic']}")

Loaded 31 chunks
Sample: achievements_chunk_0 — Introduction to Aethon Dynamics


## 2 — Generate synthetic testset (gold labels)

For each chunk → LLM writes one specific question + reference answer.
That chunk becomes the gold doc for that query.

Saved to `testset.json` so you only pay for this API call once.
Re-run only if chunks change.

In [3]:
TESTSET_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "testset_item",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "question":  {"type": "string", "description": "A specific question answered ONLY by this passage."},
                "answer":    {"type": "string", "description": "Concise reference answer grounded in the passage."}
            },
            "required": ["question", "answer"],
            "additionalProperties": False
        }
    }
}

def generate_testset(chunks: list[dict], min_chars: int = 150) -> list[dict]:
    testset = []
    eligible = [c for c in chunks if len(c["text"]) >= min_chars]
    print(f"Generating questions for {len(eligible)} chunks...")
    for c in eligible:
        resp = oai.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content":
                 "Given a passage, write ONE specific question that is answered ONLY by it "
                 "(not by any other passage), plus a concise reference answer."},
                {"role": "user", "content": f"PASSAGE:\n{c['text']}"},
            ],
            temperature=0,
            max_tokens=300,
            response_format=TESTSET_SCHEMA,
        )
        obj = json.loads(resp.choices[0].message.content)
        testset.append({
            "query":     obj["question"],
            "reference": obj["answer"],
            "gold_id":   c["chunk_id"],
            "gold_topic": c["topic"],
        })
        print(f"  [{len(testset)}/{len(eligible)}] {c['topic']}")
        time.sleep(0.2)
    return testset


# Only generate if not cached
if os.path.exists(TESTSET_FILE):
    with open(TESTSET_FILE, encoding="utf-8") as f:
        testset = json.load(f)
    print(f"Loaded testset from cache: {len(testset)} items")
else:
    testset = generate_testset(chunks)
    with open(TESTSET_FILE, "w", encoding="utf-8") as f:
        json.dump(testset, f, indent=2, ensure_ascii=False)
    print(f"\nSaved {len(testset)} items to {TESTSET_FILE}")

Generating questions for 27 chunks...
  [1/27] Company Milestones
  [2/27] Awards & Recognition
  [3/27] Patents
  [4/27] Notable Press
  [5/27] Technical Achievements
  [6/27] Company Overview
  [7/27] Company Mission
  [8/27] Product Offerings
  [9/27] Company Facts
  [10/27] Company Structure
  [11/27] Dr. Maya Krishnan
  [12/27] Daniel Osei
  [13/27] Priya Venkatesh
  [14/27] Marcus Bell
  [15/27] Sofia Reyes
  [16/27] Tom Whitaker
  [17/27] Selected Key Employees
  [18/27] GridSense Product Details
  [19/27] VoltCore Product Details
  [20/27] PulseAPI Product Details
  [21/27] Aethon Grid Suite Overview
  [22/27] Overview of Aethon Dynamics
  [23/27] Revenue Growth and Breakdown
  [24/27] Key Customers of Aethon
  [25/27] Sales Strategy and Cycle
  [26/27] Sales Pipeline Overview
  [27/27] Market Regions and Growth

Saved 27 items to ./testset.json


### Inspect a few testset items

In [4]:
for t in testset[:3]:
    print(f"Gold chunk : {t['gold_id']}")
    print(f"Topic      : {t['gold_topic']}")
    print(f"Query      : {t['query']}")
    print(f"Reference  : {t['reference']}")
    print()

Gold chunk : achievements_chunk_1
Topic      : Company Milestones
Query      : What significant event occurred for Aethon Dynamics in 2022?
Reference  : In 2022, Aethon Dynamics signed its first six-figure GridSense contract with Cascade Power & Light and closed its Series B funding.

Gold chunk : achievements_chunk_2
Topic      : Awards & Recognition
Query      : What award did VoltCore receive in 2023?
Reference  : VoltCore received the 2023 GridEdge Innovation Award.

Gold chunk : achievements_chunk_3
Topic      : Patents
Query      : What is the title of the US Patent granted in 2024?
Reference  : Low-latency market-data reconciliation for distributed energy trading.



## 3 — Retrievers
Same three modes as the rewrite notebook — copied here so this notebook is self-contained.

In [5]:
def embed(text: str) -> list[float]:
    v = np.array(
        oai.embeddings.create(input=[text], model=EMBED_MODEL).data[0].embedding,
        dtype=np.float32
    )
    return (v / np.linalg.norm(v)).tolist()


def chroma_query(query_vec: list[float], n: int = TOP_K) -> list[dict]:
    """Raw Chroma query, returns list of {chunk_id, text, filename, topic, score}."""
    r = collection.query(
        query_embeddings=[query_vec],
        n_results=n,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "chunk_id": r["ids"][0][i],
            "text":     r["documents"][0][i],
            "filename": r["metadatas"][0][i]["filename"],
            "topic":    r["metadatas"][0][i]["topic"],
            "score":    round(1 - r["distances"][0][i], 4),
        }
        for i in range(len(r["ids"][0]))
    ]


def retrieve_base(query: str) -> list[dict]:
    return chroma_query(embed(query))


def retrieve_hyde(query: str) -> list[dict]:
    hypo = oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "Write a short factual passage (3-5 sentences) that directly answers the question. Use language typical of a company knowledge base."},
            {"role": "user",   "content": query},
        ],
        temperature=0, max_tokens=200,
    ).choices[0].message.content.strip()
    return chroma_query(embed(hypo))


def retrieve_multi(query: str, n_variants: int = 3) -> list[dict]:
    variants_raw = oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": f"Generate {n_variants} alternative phrasings of the search query. One per line, no numbering."},
            {"role": "user",   "content": query},
        ],
        temperature=0.4, max_tokens=200,
    ).choices[0].message.content.splitlines()
    queries = [query] + [v.strip() for v in variants_raw if v.strip()][:n_variants]

    pool: dict[str, dict] = {}
    for q in queries:
        for hit in chroma_query(embed(q)):
            if hit["chunk_id"] not in pool or hit["score"] > pool[hit["chunk_id"]]["score"]:
                pool[hit["chunk_id"]] = hit
    return sorted(pool.values(), key=lambda x: x["score"], reverse=True)

## 4 — Retrieval metrics (pure math, no LLM)

- **MRR** — where does the gold chunk first appear? 1/rank. Perfect=1.0, never found=0.
- **nDCG@k** — like MRR but rewards finding the gold chunk higher in the list more heavily (log discount).
- **Hit@k** — binary: did the gold chunk appear anywhere in top-k?
- **Recall@k** — same as Hit@k for single gold doc (fraction of gold found in top-k).

In [6]:
def reciprocal_rank(ranked_ids: list[str], gold_id: str) -> float:
    for i, cid in enumerate(ranked_ids, 1):
        if cid == gold_id:
            return 1.0 / i
    return 0.0


def ndcg_at_k(ranked_ids: list[str], gold_id: str, k: int) -> float:
    dcg  = sum(1.0 / math.log2(i + 2)
               for i, cid in enumerate(ranked_ids[:k]) if cid == gold_id)
    idcg = 1.0 / math.log2(2)   # ideal: gold at rank 1
    return dcg / idcg if idcg else 0.0


def hit_at_k(ranked_ids: list[str], gold_id: str, k: int) -> float:
    return 1.0 if gold_id in ranked_ids[:k] else 0.0


def mean_metrics(per_query: list[dict]) -> dict:
    keys = per_query[0].keys()
    return {k: round(sum(d[k] for d in per_query) / len(per_query), 4) for k in keys}


def eval_retriever(name: str, retrieve_fn, testset: list[dict], k: int = EVAL_K) -> dict:
    """
    Run retrieve_fn over every testset query, compute per-query metrics,
    return mean scores.
    """
    per_query = []
    print(f"Evaluating [{name}] over {len(testset)} queries...", end=" ")
    for t in testset:
        hits       = retrieve_fn(t["query"])
        ranked_ids = [h["chunk_id"] for h in hits]
        per_query.append({
            f"mrr":        reciprocal_rank(ranked_ids, t["gold_id"]),
            f"ndcg@{k}":   ndcg_at_k(ranked_ids, t["gold_id"], k),
            f"hit@{k}":    hit_at_k(ranked_ids, t["gold_id"], k),
        })
        time.sleep(0.1)
    means = mean_metrics(per_query)
    print("done")
    return {"mode": name, **means}

### Run retrieval eval across all three modes
This makes API calls (embed + rewrite) for every query × mode. ~5 min for 31 queries.

In [7]:
retrieval_results = [
    eval_retriever("Base",        retrieve_base,  testset),
    eval_retriever("HyDE",        retrieve_hyde,  testset),
    eval_retriever("Multi-query", retrieve_multi, testset),
]

# Print results table
header = f"{'Mode':<20} {'MRR':>8} {'nDCG@'+str(EVAL_K):>10} {'Hit@'+str(EVAL_K):>8}"
print()
print(header)
print("-" * len(header))
for r in retrieval_results:
    print(f"{r['mode']:<20} {r['mrr']:>8.4f} {r[f'ndcg@{EVAL_K}']:>10.4f} {r[f'hit@{EVAL_K}']:>8.4f}")

Evaluating [Base] over 27 queries... 

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


done
Evaluating [HyDE] over 27 queries... done
Evaluating [Multi-query] over 27 queries... done

Mode                      MRR     nDCG@5    Hit@5
-------------------------------------------------
Base                   0.9704     0.9773   1.0000
HyDE                   0.8753     0.8944   0.9630
Multi-query            0.9722     0.9789   1.0000


## 5 — LLM judge (generation track)

For each query we:
1. Retrieve top-k chunks (using best retriever from above)
2. Generate an answer from those chunks
3. Ask the LLM judge to score on three axes using structured outputs:

| Metric | What it checks |
|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved chunks? (hallucination check) |
| **Answer relevancy** | Does the answer actually address the question? |
| **Context precision** | Are the retrieved chunks relevant to the question? (retrieval quality from generation's POV) |

In [8]:
JUDGE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "judge_scores",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "faithfulness": {
                    "type": "integer",
                    "description": "0-10. Every claim in the answer is supported by the context. 10=fully grounded, 0=hallucinated."
                },
                "answer_relevancy": {
                    "type": "integer",
                    "description": "0-10. The answer directly addresses the question. 10=perfectly on-topic, 0=completely off-topic."
                },
                "context_precision": {
                    "type": "integer",
                    "description": "0-10. The retrieved context contains information needed to answer the question. 10=highly relevant, 0=useless."
                },
                "faithfulness_reason":     {"type": "string", "description": "One sentence explaining the faithfulness score."},
                "answer_relevancy_reason": {"type": "string", "description": "One sentence explaining the answer relevancy score."},
                "context_precision_reason":{"type": "string", "description": "One sentence explaining the context precision score."}
            },
            "required": [
                "faithfulness", "answer_relevancy", "context_precision",
                "faithfulness_reason", "answer_relevancy_reason", "context_precision_reason"
            ],
            "additionalProperties": False
        }
    }
}


def generate_answer(question: str, chunks: list[dict]) -> str:
    context = "\n\n".join(c["text"] for c in chunks)
    return oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content":
             "Answer the question using ONLY the provided context. "
             "If the context does not contain the answer, say so explicitly."},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
        ],
        temperature=0, max_tokens=512,
    ).choices[0].message.content.strip()


def llm_judge(question: str, context_chunks: list[dict],
              answer: str, reference: str) -> dict:
    prompt = (
        f"QUESTION: {question}\n\n"
        f"REFERENCE ANSWER: {reference}\n\n"
        f"RETRIEVED CONTEXT:\n" +
        "\n---\n".join(c["text"][:400] for c in context_chunks) +
        f"\n\nGENERATED ANSWER: {answer}"
    )
    resp = oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content":
             "You are a strict RAG evaluation judge. Score the answer on faithfulness, "
             "answer relevancy, and context precision. Be critical — only award high scores "
             "when clearly deserved."},
            {"role": "user", "content": prompt},
        ],
        temperature=0, max_tokens=600,
        response_format=JUDGE_SCHEMA,
    )
    return json.loads(resp.choices[0].message.content)

### Run generation eval
Uses the best retriever (multi-query) for context. Runs over all testset queries.

In [9]:
gen_results = []
print(f"Running generation eval over {len(testset)} queries...\n")

for i, t in enumerate(testset):
    hits    = retrieve_multi(t["query"])[:5]          # top-5 chunks as context
    answer  = generate_answer(t["query"], hits)
    scores  = llm_judge(t["query"], hits, answer, t["reference"])

    gen_results.append({
        "query":              t["query"],
        "gold_topic":         t["gold_topic"],
        "answer":             answer,
        "faithfulness":       scores["faithfulness"],
        "answer_relevancy":   scores["answer_relevancy"],
        "context_precision":  scores["context_precision"],
        "faith_reason":       scores["faithfulness_reason"],
        "relevancy_reason":   scores["answer_relevancy_reason"],
        "precision_reason":   scores["context_precision_reason"],
    })
    print(f"  [{i+1}/{len(testset)}] F={scores['faithfulness']} "
          f"AR={scores['answer_relevancy']} "
          f"CP={scores['context_precision']}  — {t['gold_topic']}")
    time.sleep(0.3)

print("\ndone")

Running generation eval over 27 queries...

  [1/27] F=10 AR=10 CP=10  — Company Milestones
  [2/27] F=10 AR=10 CP=10  — Awards & Recognition
  [3/27] F=10 AR=10 CP=10  — Patents
  [4/27] F=10 AR=10 CP=10  — Notable Press
  [5/27] F=10 AR=10 CP=10  — Technical Achievements
  [6/27] F=10 AR=10 CP=10  — Company Overview
  [7/27] F=10 AR=10 CP=10  — Company Mission
  [8/27] F=10 AR=10 CP=10  — Product Offerings
  [9/27] F=10 AR=10 CP=10  — Company Facts
  [10/27] F=10 AR=10 CP=10  — Company Structure
  [11/27] F=10 AR=10 CP=10  — Dr. Maya Krishnan
  [12/27] F=10 AR=10 CP=10  — Daniel Osei
  [13/27] F=10 AR=10 CP=10  — Priya Venkatesh
  [14/27] F=10 AR=10 CP=10  — Marcus Bell
  [15/27] F=10 AR=10 CP=10  — Sofia Reyes
  [16/27] F=10 AR=10 CP=10  — Tom Whitaker
  [17/27] F=10 AR=10 CP=10  — Selected Key Employees
  [18/27] F=10 AR=10 CP=10  — GridSense Product Details
  [19/27] F=10 AR=10 CP=10  — VoltCore Product Details
  [20/27] F=10 AR=10 CP=10  — PulseAPI Product Details
  [21/27] F=10 

## 6 — Final results tables

In [10]:
# ── Retrieval table ────────────────────────────────────────────────────────────
print("RETRIEVAL EVAL")
print("=" * 50)
header = f"{'Mode':<20} {'MRR':>8} {'nDCG@'+str(EVAL_K):>10} {'Hit@'+str(EVAL_K):>8}"
print(header)
print("-" * 50)
for r in retrieval_results:
    print(f"{r['mode']:<20} {r['mrr']:>8.4f} {r[f'ndcg@{EVAL_K}']:>10.4f} {r[f'hit@{EVAL_K}']:>8.4f}")

print()

# ── Generation table ───────────────────────────────────────────────────────────
def avg(key): return round(sum(r[key] for r in gen_results) / len(gen_results), 2)

print("GENERATION EVAL (LLM judge, scores 0-10)")
print("=" * 50)
print(f"{'Metric':<25} {'Mean score':>10}")
print("-" * 50)
print(f"{'Faithfulness':<25} {avg('faithfulness'):>10}")
print(f"{'Answer relevancy':<25} {avg('answer_relevancy'):>10}")
print(f"{'Context precision':<25} {avg('context_precision'):>10}")

print()

# ── Per-query generation detail ────────────────────────────────────────────────
print("PER-QUERY GENERATION BREAKDOWN")
print("=" * 90)
print(f"{'Topic':<35} {'F':>4} {'AR':>4} {'CP':>4}  Faithfulness reason")
print("-" * 90)
for r in gen_results:
    topic = r['gold_topic'][:33]
    print(f"{topic:<35} {r['faithfulness']:>4} {r['answer_relevancy']:>4} {r['context_precision']:>4}  {r['faith_reason'][:60]}")

RETRIEVAL EVAL
Mode                      MRR     nDCG@5    Hit@5
--------------------------------------------------
Base                   0.9704     0.9773   1.0000
HyDE                   0.8753     0.8944   0.9630
Multi-query            0.9722     0.9789   1.0000

GENERATION EVAL (LLM judge, scores 0-10)
Metric                    Mean score
--------------------------------------------------
Faithfulness                    10.0
Answer relevancy                10.0
Context precision               10.0

PER-QUERY GENERATION BREAKDOWN
Topic                                  F   AR   CP  Faithfulness reason
------------------------------------------------------------------------------------------
Company Milestones                    10   10   10  The answer accurately reflects the significant events for Ae
Awards & Recognition                  10   10   10  The answer accurately reflects the information provided in t
Patents                               10   10   10  The answer accuratel

## 7 — Save results

In [11]:
output = {
    "retrieval": retrieval_results,
    "generation": {
        "mean": {
            "faithfulness":      avg("faithfulness"),
            "answer_relevancy":  avg("answer_relevancy"),
            "context_precision": avg("context_precision"),
        },
        "per_query": gen_results,
    }
}
with open("./eval_results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Results saved to eval_results.json")

Results saved to eval_results.json
